# Setup

In [1]:
%load_ext autoreload
%autoreload 2
%config InlineBackend.figure_format = "retina"

In [2]:
import os
import sys

# so that mllm_shap can be imported without installing the package
sys.path.insert(0, os.path.abspath("../mllm_shap/src"))

os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["TQDM_DISABLE"] = "1"
os.environ["LOG_LEVEL"] = "INFO"

In [3]:
import numpy as np
import pandas as pd
import torch

np.random.seed(42)

device = torch.device("mps") if torch.backends.mps.is_available() else torch.device("cpu")
print(f"Using device: {device}")

Using device: mps


In [4]:
from mllm_shap.connectors import LiquidAudio, ModelConfig
from mllm_shap.connectors.enums import ModelHistoryTrackingMode, Role, SystemRolesSetup
from mllm_shap.connectors.filters import KeepAllTokens
from mllm_shap.utils.jupyter import display_shap_colors_df
from mllm_shap.shap import ComplementaryNeymanShapExplainer, Explainer
from mllm_shap.shap.normalizers import MinMaxNormalizer

Define LiquidAudio model (this call loads it up to the memory!).

In [5]:
model = LiquidAudio(
    device=device, history_tracking_mode=ModelHistoryTrackingMode.TEXT
)  # track and generate only text history

W1111 00:13:10.864000 55518 torch/distributed/elastic/multiprocessing/redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


Create explainer that will make initial call and then explain it using shapley values using Neyman Formula.

In [6]:
explainer = Explainer(model=model, shap_explainer=ComplementaryNeymanShapExplainer(normalizer=MinMaxNormalizer(), initial_num_samples=2, num_samples=200))

Create new chat instance and assign it messages. ComplementaryNeymanShapExplainer currently supports only `SystemRolesSetup.SYSTEM_ASSISTANT` mode. It requires at least one assistant turn to be present.

In [7]:
chat = model.get_new_chat(
    system_roles_setup=SystemRolesSetup.SYSTEM_ASSISTANT,  # calculate shapley values for all roles
    token_filter=KeepAllTokens(),  # keep all tokens for shapley values calculation
)

chat.new_turn(Role.ASSISTANT)
chat.add_text("Be helpful and concise.")
chat.end_turn()

chat.new_turn(Role.USER)
chat.add_text("Who are you? Where have been created?")
chat.end_turn()

# Usage

Let's calculate shapley values for current conversation.

Generation kwargs allows to customize model interference - here we limit it to 4 tokens and change text_temperature from default 0.0 to 0.2, text_top_k from default 1 to 3. 

In [8]:
generation_kwargs = {"max_new_tokens": 4, "model_config": ModelConfig(text_temperature=0.2, text_top_k=3)}

result = explainer(
    chat=chat,
    generation_kwargs=generation_kwargs,
    progress_bar=True,  # show progress bar during generation, default
)

2025-11-11 00:13:15,013 - mllm_shap.shap.compact - INFO - Generating full response from the model...
2025-11-11 00:13:15,509 - mllm_shap.shap.base._masks_manager - INFO - Number of tokens for explainability: 9 (up to 511 additional calls)
2025-11-11 00:13:15,535 - mllm_shap.shap.neyman - INFO - Starting initial sampling step with 2 samples per entry in M


Calculating SHAP values:   0%|          | 0/200 [00:00<?, ?it/s]

2025-11-11 00:13:56,397 - mllm_shap.shap.neyman - INFO - Starting Neyman allocation step with 54 remaining samples


Calculating SHAP values:   0%|          | 0/200 [00:00<?, ?it/s]

2025-11-11 00:14:37,388 - mllm_shap.shap.neyman - WARNING - Neyman allocation ended yet sampling budget was not fully used (108/200).


Let's see final Shap values.

In [9]:
display_shap_colors_df(
    pd.DataFrame(
        list(
            zip(
                [chat.decode_text(token) for token in result.full_chat.input_tokens],
                result.full_chat.cache.normalized_values.tolist(),
            )
        ),
        columns=["Text", "Shapley Value"],
    )
)

,Text,Shapley Value
0,<|startoftext|>,nan
1,<|im_start|>,nan
2,assistant,nan
3,,nan
4,Be,nan
5,helpful,nan
6,and,nan
7,concise,nan
8,.,nan
9,<|im_end|>,nan


# Tests

Presence matrix:

In [10]:
explainer.shap_explainer._M

tensor([[ 2,  2,  7,  8, 13, 13, 17, 18, 22,  2],
        [ 2,  3,  5,  8, 16, 10, 17, 20, 21,  2],
        [ 2,  3,  7,  7,  7, 19, 18, 18, 21,  2],
        [ 2,  4,  4,  9, 11, 15, 16, 21, 20,  2],
        [ 2,  3,  3,  6, 12, 14, 19, 22, 21,  2],
        [ 2,  3,  6, 13, 10, 16, 12, 19, 21,  2],
        [ 2,  2,  7, 11,  9, 17, 14, 18, 22,  2],
        [ 2,  2,  5,  6, 14, 12, 19, 20, 22,  2],
        [ 2,  2,  6,  7, 12, 14, 18, 19, 22,  2]], device='mps:0',
       dtype=torch.int16)

CC matrices:

In [11]:
explainer.shap_explainer._C

tensor([[-0.5156, -0.1914,  0.1719, -0.3906,  0.0547,  1.3203,  1.4375,  2.4375,
          3.7969,  0.5156],
        [-0.5156, -0.6797, -0.4492, -0.3672, -0.7227,  0.5469,  1.4531,  1.8125,
          3.3438,  0.5156],
        [-0.5156, -0.3633,  0.1055,  0.1406,  0.3438,  1.6094,  1.9688,  2.3438,
          3.6406,  0.5156],
        [-0.5156, -0.4375, -0.6250, -0.4023, -0.6328,  0.6367,  1.4219,  1.6250,
          3.5625,  0.5156],
        [-0.5156, -0.5703, -0.4609, -0.8633, -0.3672,  0.9023,  0.9688,  1.7969,
          3.4219,  0.5156],
        [-0.5156, -0.6562, -0.8477, -1.5000, -1.0391,  0.2305,  0.3320,  1.4062,
          3.3281,  0.5156],
        [-0.5156, -0.3359, -1.0312, -0.7734, -0.8828,  0.3867,  1.0625,  1.2188,
          3.6719,  0.5156],
        [-0.5156, -0.3125, -0.5547, -0.5078, -1.2500,  0.0195,  1.3281,  1.7031,
          3.7031,  0.5156],
        [-0.5156, -0.4531, -0.8047, -0.8320, -0.5820,  0.6875,  1.0000,  1.4375,
          3.5781,  0.5156]], device='mps:0', dt

In [12]:
explainer.shap_explainer._C_squared

tensor([[0.1328, 0.0195, 0.0591, 0.1055, 0.1602, 0.2295, 0.3008, 0.4023, 0.7188,
         0.1328],
        [0.1328, 0.1543, 0.0801, 0.1318, 0.2539, 0.1387, 0.2754, 0.3789, 0.5859,
         0.1328],
        [0.1328, 0.0439, 0.0703, 0.0378, 0.0923, 0.2969, 0.3691, 0.3906, 0.6953,
         0.1328],
        [0.1328, 0.0479, 0.1055, 0.1055, 0.1084, 0.2852, 0.3008, 0.3594, 0.6914,
         0.1328],
        [0.1328, 0.1162, 0.0786, 0.1514, 0.2031, 0.1865, 0.2559, 0.3828, 0.6250,
         0.1328],
        [0.1328, 0.1465, 0.1494, 0.2305, 0.1650, 0.2266, 0.1768, 0.3086, 0.5938,
         0.1328],
        [0.1328, 0.0608, 0.1631, 0.1562, 0.1289, 0.2617, 0.2500, 0.2969, 0.6797,
         0.1328],
        [0.1328, 0.0488, 0.0928, 0.1299, 0.2373, 0.1533, 0.2773, 0.3711, 0.6914,
         0.1328],
        [0.1328, 0.1025, 0.1221, 0.1729, 0.2148, 0.1768, 0.2344, 0.3398, 0.6367,
         0.1328]], device='mps:0', dtype=torch.bfloat16)